# NeuroFusionAI: Step 1 - Multimodal Data Preprocessing
This notebook walks through the data preparation and preprocessing pipeline of the three modalities:
1. **Hand-drawn Images** (resizing to 224x224 and splitting).
2. **Acoustic Voice Classification features** (feature standardization, duplicate removal, and stratified splitting).
3. **Tabular UPDRS Progression Telemonitoring features** (standardization and splitting).
We will import our modular preprocessing python package to execute the preprocessing.


In [ ]:
import os
import sys
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
import glob

# Resolve project root (walk up until we find the 'datasets' directory)
PROJECT_ROOT = os.path.abspath('.')
for p in [os.path.abspath(x) for x in sys.path if x]:
    if os.path.isdir(os.path.join(p, 'datasets')):
        PROJECT_ROOT = p
        break
    elif os.path.isdir(os.path.join(os.path.dirname(p), 'datasets')):
        PROJECT_ROOT = os.path.dirname(p)
        break
while not os.path.isdir(os.path.join(PROJECT_ROOT, 'datasets')) and PROJECT_ROOT != os.path.dirname(PROJECT_ROOT):
    PROJECT_ROOT = os.path.dirname(PROJECT_ROOT)
os.chdir(PROJECT_ROOT)
sys.path.insert(0, PROJECT_ROOT)
print('Project root:', PROJECT_ROOT)


## 1. Run Preprocessing Pipelines
We execute the preprocessing functions that resize images, standardize tabular features, handle missing data, and create stratified splits.


In [ ]:
from preprocessing.image_preprocessing import split_and_preprocess_images
from preprocessing.mri_preprocessing import split_and_preprocess_mri
from preprocessing.voice_preprocessing import preprocess_voice
from preprocessing.telemonitor_preprocessing import preprocess_telemonitor

print('Running Spiral Image Preprocessing...')
split_and_preprocess_images('datasets/raw_spiral', 'datasets/processed/images')

print('\nRunning MRI Preprocessing...')
split_and_preprocess_mri('datasets/mri', 'datasets/processed/mri')

print('\nRunning Oxford Voice Preprocessing...')
preprocess_voice('datasets/voice/oxford/parkinsons.data', 'datasets')

print('\nRunning Telemonitoring Preprocessing...')
preprocess_telemonitor('datasets/voice/telemonitoring/parkinsons_updrs.data', 'datasets')


## 2. Visualize Preprocessed Drawings and Verify MRI
Let's display some control (normal) vs Parkinson drawings, and check the MRI count.


In [ ]:
normal_images = glob.glob('datasets/train/images/normal/*.png')
parkinson_images = glob.glob('datasets/train/images/parkinson/*.png')

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
if normal_images:
    axes[0].imshow(Image.open(normal_images[0]))
    axes[0].set_title('Normal Control Drawing')
    axes[0].axis('off')
if parkinson_images:
    axes[1].imshow(Image.open(parkinson_images[0]))
    axes[1].set_title('Parkinson Drawing')
    axes[1].axis('off')
plt.show()

# Check MRI images if present
normal_mri = glob.glob('datasets/train/mri/normal/*.png') + glob.glob('datasets/train/mri/normal/*.jpg')
parkinson_mri = glob.glob('datasets/train/mri/parkinson/*.png') + glob.glob('datasets/train/mri/parkinson/*.jpg')
print(f'Found {len(normal_mri)} normal MRIs and {len(parkinson_mri)} Parkinson MRIs in train split.')


## 3. Verify Voice Acoustics Split Shape & Stats
Let's check the size and content of our voice splits.


In [ ]:
train_voice = pd.read_csv('datasets/train/voice/oxford_train.csv')
print('Voice Train split shape:', train_voice.shape)
print('Distribution of Parkinson Status (1: Parkinson, 0: Healthy):')
print(train_voice['status'].value_counts(normalize=True))
train_voice.head(3)


## 4. Verify Telemonitoring UPDRS Split Shape & Stats
Let's inspect the tabular data and regression targets (`motor_UPDRS` and `total_UPDRS`) for severity tracking.


In [ ]:
train_tele = pd.read_csv('datasets/train/telemonitoring/telemonitor_train.csv')
print('Telemonitoring Train split shape:', train_tele.shape)
print('Target statistics for severity assessment:')
print(train_tele[['motor_UPDRS', 'total_UPDRS']].describe())
train_tele.head(3)
